# DB-QSP
https://arxiv.org/pdf/2504.01077

In [115]:
from qrisp import *
from qrisp.operators import X, Y, Z
from qrisp.jasp import q_fori_loop, q_cond, check_for_tracing_mode
from jax import lax
import scipy as sp
import numpy as np
import jax.numpy as jnp
import pytest

## Example model: XXZ

In [116]:
from qrisp.vqe.problems.heisenberg import create_heisenberg_init_function, heisenberg_problem, create_heisenberg_hamiltonian
L = 5
G = nx.Graph()
G.add_edges_from([(k,(k+1)%L) for k in range(L-1)]) 
J = 1
B = 0.5
H = create_heisenberg_hamiltonian(G, J, B)
print(H)

X(0)*X(1) + X(1)*X(2) + X(2)*X(3) + X(3)*X(4) + Y(0)*Y(1) + Y(1)*Y(2) + Y(2)*Y(3) + Y(3)*Y(4) + 0.5*Z(0) + Z(0)*Z(1) + 0.5*Z(1) + Z(1)*Z(2) + 0.5*Z(2) + Z(2)*Z(3) + 0.5*Z(3) + Z(3)*Z(4) + 0.5*Z(4)


In [117]:
# Define scaling factor
F = 1

def exp_H(qv, t):
    H.trotterization(method='commuting')(qv,t/F,5)

# Hamiltonian simulation via second order Suzuki-Trotter formula with 2 steps
def exp_H_2(qv, t):
    H.trotterization(order=2,method='commuting')(qv,t/F,2)

# Calculate E and V

In [ ]:
# in qrisp
def calculate_EV(H, qv):
    state_prep = lambda : qv
    H_2 = H**2
    E = H.expectation_value(state_prep, diagonalisation_method="commuting")()
    E_2 = H_2.expectation_value(state_prep, diagonalisation_method="commuting")()
    
    V = E_2 - E**2
    
    return E, V

# matrix, for tests only
def compute_moments(psi, H):
    psi = np.array([psi]).transpose()
    E = (psi.conj().T @ H @ psi)[0,0].real
    S = (psi.conj().T @ H @ H @ psi)[0,0].real
    return E, S, S - E**2

In [ ]:
# example
qv = QuantumVariable(L)
x(qv[1])
E, V = calculate_EV(H, qv)
print(E, V)

# matrix
psi = np.zeros(2**L)
psi[2**(L-2)] = 1
psi = psi/np.linalg.norm(psi)
H_matrix = H.to_array()
E, _, V = compute_moments(psi, H_matrix)
print(E, V)
# expect the same results for both

Simulating 5 qubits.. |                                                      | [  0%]

1.4985092049561903 8.000373743972375                                                 
1.5 8.0


## calculate s and phase
$$
\frac{(H-zI)\ket{\Psi}}{\|(H-zI)\ket\Psi\|}   =e^{i\theta\Psi}e^{s_{\Psi}[\Psi, H]}\ket\Psi.
$$

with $s_k = \frac{-1}{\sqrt{V_k}}\arccos\left(\frac{|E_{k}-z_{k}|}{\sqrt{V_{k}+|E_{k}-z_{k}|^2}}\right)$
and $\theta_k = \arg\left(\frac{E_k-z_k}{|E_k-z_k|}\right).$

In [121]:
def QSP_unitary_synthesis_params(E, V, z):
    diff = E - z
    s = -1/jnp.sqrt(V)*jnp.arccos(jnp.abs(diff)/jnp.sqrt(V+jnp.abs(diff)**2))
    theta = jnp.angle(diff)
    return s, theta

## DB-QSP steps
$$
\frac{(H-zI)\ket{\Psi}}{\|(H-zI)\ket\Psi\|}   =e^{i\theta\Psi}e^{s_{\Psi}[\Psi, H]}\ket\Psi.
$$
$$
e^{s_\Psi[\Psi,H]} = \left(
e^{is_\Psi^{(N)} \Psi}e^{is_\Psi^{(N)} H}
e^{-is_\Psi^{(N)} \Psi}e^{-is_\Psi^{(N)} H}
\right)^N \nonumber+O(s_\Psi^{3/2}/\sqrt N)\ , 
$$


### 1 step

In [174]:
def DB_QSP(qarg, U0, H, exp_H, z, N):
    U0(qarg)
    def conjugator(qarg):
        with invert():
            # here U0 needs to gnerate Psi
            U0(qarg)
            
    def reflection(qarg, t_):
        with conjugate(conjugator)(qarg):
            if isinstance(qarg,QuantumArray):
                qubits = sum([qv.reg for qv in qarg.flatten()], [])
                mcp(t_, qubits, ctrl_state=0, method="khattar")
            else:
                mcp(t_, qarg, ctrl_state=0, method="khattar")
    E, V = calculate_EV(H, qarg)
    s, theta = QSP_unitary_synthesis_params(E, V, z)
    s_ = jnp.sqrt(jnp.abs(s)/N)
    for _ in range(N):
        exp_H(qarg, s_)
        reflection(qarg, s_)
        exp_H(qv, -s_)
        reflection(qarg, -s_)
        reflection(qarg, -theta)
    return s, theta

In [175]:
# example numpy calculation
psi = np.zeros(2**L)
psi[2**(L-2)] = 1
psi = psi/np.linalg.norm(psi)
H_matrix = H.to_array()
psi_dm = np.outer(psi, psi.conj())

zk = -0.2

# target state
I = np.eye(2**L, 2**L)
psi_target = (H_matrix-zk*I)@ psi
psi_target /= np.linalg.norm(psi_target)

# db-qsp state
E, _, V = compute_moments(psi, H_matrix)
print("Initial EV", (E, V))
s, theta = QSP_unitary_synthesis_params(E, V, zk)
print("     s, theta", (s, theta))
psi_qsp = sp.linalg.expm(1j*theta*psi_dm) @ sp.linalg.expm(s*(psi_dm@H_matrix-H_matrix@psi_dm)) @ psi
print("     Fidelity", abs(np.vdot(psi_target, psi_qsp))**2)
E_qsp = np.vdot(psi_qsp, H_matrix @ psi_qsp).real
V_qsp = np.vdot(psi_qsp, H_matrix @ H_matrix @ psi_qsp).real - E_qsp**2
print("After DB-QSP", (E_qsp, V_qsp))

Initial EV (np.float64(1.5), np.float64(8.0))
     s, theta (Array(-0.36402278, dtype=float64), Array(0., dtype=float64))
     Fidelity 0.9999999999999998
After DB-QSP (np.float64(4.732323232323233), np.float64(2.988266503418018))


In [176]:
# example
qv = QuantumVariable(L)
def U0(qv):
    x(qv[1])
U0(qv)
print("Initial EV", calculate_EV(H, qv))
zk = -0.2
qv = QuantumVariable(L)
s, theta = DB_QSP(qv, U0, H, exp_H, zk, 1)
print("     s, theta", (s, theta))
print("After DB-QSP", calculate_EV(H, qv))

Initial EV (1.510548078140163, 7.965833840679403)                                    
     s, theta (Array(-0.36363934, dtype=float64, weak_type=True), Array(0., dtype=float64))2K
After DB-QSP (1.8656916290263732, 8.05104861318811)                                  


## multiple steps

$$
\frac{p(H)\ket{\Psi_0}}{\|p(H)\ket{\Psi_0}\|}=\prod_{k=0}^{K-1} e^{i \theta_k \Psi_{k}}  e^{s_{k}[\Psi_{k},H]}\ket{\Psi_0},
$$

In [126]:
def run_DBQSP(U_0, H, exp_H, z_list, N):
    start = time.time()

    L = H.find_minimal_qubit_amount()
    qv = QuantumVariable(L)
    U_0(qv)
    E, V = calculate_EV(H, qv)
    circuits = [qv.qs.compile()]
    energies = [E]
    variances = [V]
    thetas = []
    ss = []
    runtimes = []
    
    for z in z_list:
        s, theta = DB_QSP(qv, H, exp_H, z, N)
        E, V = calculate_EV(H, qv)
        circuits.append(qv.qs.compile())
        energies.append(E)
        variances.append(V)
        thetas.append(theta)
        ss.append(s)
        end = time.time()
        runtimes.append(end-start)
    result_dict = {'s':ss, 'theta':thetas, 'energies':energies, 'variances': variances, 'circuits':circuits, 'runtimes':runtimes}
    return result_dict

In [129]:
z_list = [0.5, -0.2, 0.1+0.1j]
def U_0(qv):
    x(qv[1])
    return qv
results = run_DBQSP(U_0, H, exp_H, z_list, N=1)

In [130]:
results['energies']

[1.5018564617526686, 1.215371674567496, 1.0800104960054662, 1.227111631649033]

In [172]:
pH = [I]
for k, zk in enumerate(z_list):
    pH.append((H_matrix - zk * I)@pH[-1])
energy_expect = lambda U: (np.vdot(U@psi, H_matrix@U@psi)/np.linalg.norm(U@psi)**2).real
energies = [energy_expect(U) for U in pH]
print(energies)

[np.float64(1.5), np.float64(4.166666666666666), np.float64(5.1789244760775), np.float64(5.318515519580622)]
